# 📄 PDF Table Extractor — v2 (Improved)
### Extracts tables from scanned PDFs → clean Excel / CSV  
**Works fully offline. No internet required.**

---
### 📋 One-time Setup
```
pip install pymupdf pytesseract Pillow openpyxl opencv-python-headless pdfplumber numpy pandas
```
**Tesseract OCR binary:**
- Windows: https://github.com/UB-Mannheim/tesseract/wiki → install → add to PATH
- Linux: `sudo apt install tesseract-ocr`
- macOS: `brew install tesseract`

---


In [ ]:
# ============================================================
#  ✏️  CONFIGURATION  —  Only edit this cell
# ============================================================

# Full path to your input PDF
INPUT_PDF = r"C:/Users/Raghunandan/OneDrive/Desktop/Data_Engineer/Generative AI/single_page_output1.pdf"

# Output file path — must include filename + extension (.xlsx or .csv)
# Set to None to auto-save in the same folder as the PDF
OUTPUT_FILE = r"C:/Users/Raghunandan/OneDrive/Desktop/Data_Engineer/Generative AI/Image_AI_application/PDF_Extractor/output.xlsx"

# Output format: "xlsx" or "csv"
OUTPUT_FORMAT = "xlsx"

# Pages to process: "all"  |  "1,3,5"  |  "2-6"
PAGES = "all"

# DPI for rendering (300 = normal, 400-500 = better for small/poor text)
DPI = 400

# Tesseract path — set if Tesseract is NOT in your system PATH
# Leave as None if it's already in PATH
TESSERACT_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# ============================================================
print("✅ Configuration loaded.")
print(f"   Input  : {INPUT_PDF}")
print(f"   Output : {OUTPUT_FILE}")
print(f"   Format : {OUTPUT_FORMAT.upper()}")
print(f"   Pages  : {PAGES} | DPI: {DPI}")


In [ ]:
# ============================================================
#  📦  Imports & Dependency Check
# ============================================================
import sys, os, re, csv, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")

REQUIRED = [
    ("fitz",        "pymupdf"),
    ("pytesseract", "pytesseract"),
    ("PIL",         "Pillow"),
    ("openpyxl",    "openpyxl"),
    ("cv2",         "opencv-python-headless"),
    ("pdfplumber",  "pdfplumber"),
    ("pandas",      "pandas"),
]
missing = [pip for pkg, pip in REQUIRED if not __import__("importlib").util.find_spec(pkg)]
if missing:
    print("❌ Install these and restart kernel:")
    print("   pip install " + " ".join(missing))
    raise SystemExit()

import fitz
import pytesseract
import cv2
import pandas as pd
import pdfplumber
from PIL import Image, ImageFilter, ImageEnhance
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

if TESSERACT_PATH:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

print("✅ All dependencies loaded.")


In [ ]:
# ============================================================
#  🔧  Core Functions  (do not edit)
# ============================================================

def parse_pages(spec, total):
    if spec.lower() == "all":
        return list(range(total))
    pages = set()
    for part in spec.split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            pages.update(range(int(a)-1, int(b)))
        else:
            pages.add(int(part)-1)
    return sorted(p for p in pages if 0 <= p < total)


def pdf_page_to_pil(fitz_page, dpi=400):
    """Render PDF page to PIL image at given DPI."""
    mat = fitz.Matrix(dpi/72, dpi/72)
    pix = fitz_page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def preprocess_for_ocr(pil_img):
    """
    Preprocess image for best OCR accuracy:
    - Convert to grayscale
    - Increase contrast
    - Binarize (Otsu threshold)
    - Slight denoise
    """
    img_cv = np.array(pil_img)
    gray   = cv2.cvtColor(img_cv, cv2.COLOR_RGB2GRAY)
    # Enhance contrast
    clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray   = clahe.apply(gray)
    # Denoise
    gray   = cv2.fastNlMeansDenoising(gray, h=10)
    # Otsu binarization
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(binary)


def auto_rotate_image(pil_img):
    """Detect and correct image rotation using Tesseract OSD."""
    try:
        osd = pytesseract.image_to_osd(pil_img, output_type=pytesseract.Output.DICT)
        angle = osd.get("rotate", 0)
        if angle:
            pil_img = pil_img.rotate(-angle, expand=True)
            print(f"      ↺ Auto-rotated {angle}°")
    except Exception:
        pass
    return pil_img


def detect_table_lines(pil_img):
    """
    Use OpenCV morphological operations to detect horizontal and
    vertical lines in the table grid. Returns (h_lines, v_lines).
    """
    img_cv = np.array(pil_img.convert("L"))
    _, binary = cv2.threshold(img_cv, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    w = binary.shape[1]
    h = binary.shape[0]

    # Horizontal lines
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 10, 1))
    h_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel, iterations=2)

    # Vertical lines
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 10))
    v_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel, iterations=2)

    return h_lines, v_lines


def get_row_col_boundaries(h_lines, v_lines, min_gap=10):
    """
    Extract row Y-positions and column X-positions from detected grid lines.
    Returns (row_ys, col_xs) as sorted lists.
    """
    # Row boundaries from horizontal lines
    h_proj = np.sum(h_lines, axis=1)
    row_ys  = []
    in_line = False
    for y, val in enumerate(h_proj):
        if val > h_lines.shape[1] * 0.1:
            if not in_line:
                row_ys.append(y)
                in_line = True
        else:
            in_line = False

    # Col boundaries from vertical lines
    v_proj = np.sum(v_lines, axis=0)
    col_xs  = []
    in_line = False
    for x, val in enumerate(v_proj):
        if val > v_lines.shape[0] * 0.05:
            if not in_line:
                col_xs.append(x)
                in_line = True
        else:
            in_line = False

    # Merge very close boundaries
    def merge_close(vals, gap):
        if not vals: return []
        merged = [vals[0]]
        for v in vals[1:]:
            if v - merged[-1] >= gap:
                merged.append(v)
        return merged

    return merge_close(row_ys, min_gap), merge_close(col_xs, min_gap)


def ocr_cell(pil_img, x1, y1, x2, y2, padding=4):
    """OCR a single cell region from the image."""
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(pil_img.width,  x2 + padding)
    y2 = min(pil_img.height, y2 + padding)
    cell_img = pil_img.crop((x1, y1, x2, y2))
    if cell_img.width < 5 or cell_img.height < 5:
        return ""
    text = pytesseract.image_to_string(
        cell_img,
        config="--psm 7 --oem 3 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz .,-+()/*&%#@!?:;"
    ).strip()
    return re.sub(r"\s+", " ", text)


def extract_table_grid_ocr(pil_img):
    """
    Main strategy: detect grid lines → extract cell regions → OCR each cell.
    Falls back to row-based OCR if no grid is detected.
    """
    h_lines, v_lines = detect_table_lines(pil_img)
    row_ys, col_xs = get_row_col_boundaries(h_lines, v_lines)

    print(f"      Grid detected: {len(row_ys)} rows × {len(col_xs)} cols")

    # Need at least 2 rows and 2 cols to build a grid
    if len(row_ys) >= 2 and len(col_xs) >= 2:
        grid = []
        proc_img = preprocess_for_ocr(pil_img)
        for r in range(len(row_ys) - 1):
            row_data = []
            y1 = row_ys[r]
            y2 = row_ys[r+1]
            for c in range(len(col_xs) - 1):
                x1 = col_xs[c]
                x2 = col_xs[c+1]
                text = ocr_cell(proc_img, x1, y1, x2, y2)
                row_data.append(text)
            # Only add row if it has at least one non-empty cell
            if any(cell.strip() for cell in row_data):
                grid.append(row_data)
        if grid:
            return grid

    # ── Fallback: row-band OCR ────────────────────────────────────────────────
    print("      Fallback: row-band OCR mode")
    return row_band_ocr(pil_img)


def row_band_ocr(pil_img):
    """
    Fallback: Use Tesseract word-level data to group words into rows,
    then cluster columns by x-position. Much more accurate than raw TSV.
    """
    proc_img = preprocess_for_ocr(pil_img)

    data = pytesseract.image_to_data(
        proc_img,
        output_type=pytesseract.Output.DICT,
        config="--psm 6 --oem 3"
    )

    words = []
    for i in range(len(data["text"])):
        txt  = str(data["text"][i]).strip()
        conf = int(data["conf"][i])
        if conf > 30 and txt:
            words.append({
                "text":  txt,
                "left":  data["left"][i],
                "top":   data["top"][i],
                "right": data["left"][i] + data["width"][i],
            })

    if not words:
        return []

    # ── Cluster rows by top coordinate ───────────────────────────────────────
    tops = sorted(set(w["top"] for w in words))
    row_clusters, cur = [], [tops[0]]
    for t in tops[1:]:
        if t - cur[-1] <= 15:
            cur.append(t)
        else:
            row_clusters.append(cur)
            cur = [t]
    row_clusters.append(cur)

    top_to_row = {}
    for ri, cl in enumerate(row_clusters):
        for t in cl: top_to_row[t] = ri

    for w in words:
        w["row"] = top_to_row[w["top"]]

    # ── Cluster columns by left coordinate ───────────────────────────────────
    lefts = sorted(set(w["left"] for w in words))
    col_clusters, cur = [], [lefts[0]]
    for l in lefts[1:]:
        if l - cur[-1] <= 25:
            cur.append(l)
        else:
            col_clusters.append(cur)
            cur = [l]
    col_clusters.append(cur)

    left_to_col = {}
    for ci, cl in enumerate(col_clusters):
        for l in cl: left_to_col[l] = ci

    for w in words:
        w["col"] = left_to_col[w["left"]]

    n_rows = max(w["row"] for w in words) + 1
    n_cols = max(w["col"] for w in words) + 1
    grid = [[""] * n_cols for _ in range(n_rows)]

    for w in words:
        r, c = w["row"], w["col"]
        grid[r][c] = (grid[r][c] + " " + w["text"]).strip()

    return [row for row in grid if any(cell.strip() for cell in row)]


def page_has_text(fitz_page, min_chars=80):
    """True if page has selectable text (not a scanned image)."""
    return len(fitz_page.get_text("text").strip()) > min_chars


def extract_text_page(pdf_path, page_num):
    """Extract table from a text-based PDF page using pdfplumber."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pg = pdf.pages[page_num]
            # Try structured table first
            tables = pg.extract_tables({
                "vertical_strategy":   "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance":      5,
                "intersection_tolerance": 10,
            })
            if tables:
                best = max(tables, key=lambda t: len(t) * max((len(r) for r in t), default=0))
                return [[str(cell or "").strip() for cell in row] for row in best]
            # Fallback: text lines
            text = pg.extract_text()
            if text:
                return [line.split() for line in text.splitlines() if line.strip()]
    except Exception as e:
        print(f"      pdfplumber error: {e}")
    return []

print("✅ All functions defined.")


In [ ]:
# ============================================================
#  🚀  Run Extraction
# ============================================================

pdf_path = Path(INPUT_PDF)
if not pdf_path.exists():
    raise FileNotFoundError(f"❌ File not found: {pdf_path}")

# Resolve output path
if OUTPUT_FILE:
    out_path = Path(OUTPUT_FILE)
else:
    out_path = pdf_path.with_suffix(".xlsx" if OUTPUT_FORMAT == "xlsx" else ".csv")

out_path.parent.mkdir(parents=True, exist_ok=True)

print("=" * 55)
print("  PDF Table Extractor v2")
print("=" * 55)
print(f"  Input  : {pdf_path.name}")
print(f"  Output : {out_path}")

doc    = fitz.open(str(pdf_path))
pages  = parse_pages(PAGES, len(doc))
print(f"  Pages  : {[p+1 for p in pages]} of {len(doc)} total")
print()

all_grids = []

for p in pages:
    label     = f"Page_{p+1}"
    fitz_page = doc[p]
    print(f"  [{label}]")

    if page_has_text(fitz_page):
        print("    Mode : Text-based (pdfplumber)")
        grid = extract_text_page(str(pdf_path), p)
    else:
        print("    Mode : Scanned image (OCR + grid detection)")
        pil_img = pdf_page_to_pil(fitz_page, dpi=DPI)
        pil_img = auto_rotate_image(pil_img)
        grid    = extract_table_grid_ocr(pil_img)

    rows = len(grid)
    cols = max((len(r) for r in grid), default=0)
    print(f"    Result : {rows} rows × {cols} cols")
    all_grids.append((label, grid))

doc.close()
print()
print("✅ Extraction complete — ready to save.")


In [ ]:
# ============================================================
#  💾  Save Output
# ============================================================

def save_excel(all_grids, out_path):
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    hdr_fill = PatternFill("solid", fgColor="2E75B6")
    hdr_font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
    dat_font = Font(name="Calibri", size=10)
    c_align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
    l_align  = Alignment(horizontal="left",   vertical="center", wrap_text=True)
    thin     = Side(border_style="thin", color="BFBFBF")
    brd      = Border(left=thin, right=thin, top=thin, bottom=thin)
    alt_fill = PatternFill("solid", fgColor="EBF3FB")

    for sheet_name, grid in all_grids:
        ws = wb.create_sheet(title=sheet_name[:31])
        if not grid:
            ws["A1"] = "No table data found on this page."
            continue

        n_cols = max(len(r) for r in grid)

        for ri, row in enumerate(grid, 1):
            # Pad row to full width
            row = list(row) + [""] * (n_cols - len(row))
            for ci, val in enumerate(row, 1):
                cell = ws.cell(row=ri, column=ci, value=str(val).strip() if val else "")
                cell.border = brd
                if ri == 1:
                    cell.fill      = hdr_fill
                    cell.font      = hdr_font
                    cell.alignment = c_align
                else:
                    cell.font      = dat_font
                    cell.alignment = l_align
                    if ri % 2 == 0:
                        cell.fill = alt_fill

        # Auto column widths
        for col_cells in ws.columns:
            ltr = col_cells[0].column_letter
            w   = max((len(str(c.value)) for c in col_cells if c.value), default=6)
            ws.column_dimensions[ltr].width = min(max(w + 2, 8), 40)

        ws.freeze_panes = "A2"
        ws.row_dimensions[1].height = 30

    wb.save(out_path)
    print(f"  ✅ Excel saved : {out_path}")


def save_csv(all_grids, out_path):
    base, parent = Path(out_path).stem, Path(out_path).parent
    for sheet_name, grid in all_grids:
        safe = re.sub(r"[^\w\-]", "_", sheet_name)
        fp   = parent / f"{base}_{safe}.csv"
        with open(fp, "w", newline="", encoding="utf-8-sig") as f:
            csv.writer(f).writerows(
                [str(c).strip() if c else "" for c in row] for row in grid
            )
        print(f"  ✅ CSV saved : {fp}")


if OUTPUT_FORMAT == "xlsx":
    save_excel(all_grids, str(out_path))
else:
    save_csv(all_grids, str(out_path))


In [ ]:
# ============================================================
#  👁️  Preview Extracted Data in Notebook
# ============================================================

for sheet_name, grid in all_grids:
    if not grid:
        print(f"\n[{sheet_name}] — No data.")
        continue

    n_cols = max(len(r) for r in grid)
    print(f"\n{'='*70}")
    print(f"  {sheet_name}  |  {len(grid)} rows × {n_cols} cols")
    print(f"{'='*70}")

    # Display as pandas DataFrame for nice formatting
    padded = [list(r) + [""]*(n_cols-len(r)) for r in grid]
    if len(padded) > 1:
        df = pd.DataFrame(padded[1:], columns=padded[0])
    else:
        df = pd.DataFrame(padded)

    display(df.head(30))


---
## 🗂️ Batch Mode — Process an Entire Folder of PDFs
Run the cell below to process **all PDFs in a folder** automatically.


In [ ]:
# ============================================================
#  🗂️  BATCH CONFIG  —  Edit these values
# ============================================================

BATCH_INPUT_FOLDER  = r"C:\Users\Raghunandan\Desktop\PDFs"
BATCH_OUTPUT_FOLDER = r"C:\Users\Raghunandan\Desktop\Output"
BATCH_FORMAT        = "xlsx"   # "xlsx" or "csv"
BATCH_PAGES         = "all"
BATCH_DPI           = 400

# ============================================================
import glob

pdf_files = sorted(glob.glob(os.path.join(BATCH_INPUT_FOLDER, "*.pdf")))
print(f"Found {len(pdf_files)} PDF(s)\n")

for pdf_file in pdf_files:
    pdf_path_b = Path(pdf_file)
    suffix     = ".xlsx" if BATCH_FORMAT == "xlsx" else ".csv"
    out_b      = Path(BATCH_OUTPUT_FOLDER) / pdf_path_b.with_suffix(suffix).name

    print(f"{'─'*50}")
    print(f"Processing: {pdf_path_b.name}")

    try:
        doc_b  = fitz.open(str(pdf_path_b))
        pages_b = parse_pages(BATCH_PAGES, len(doc_b))
        grids_b = []

        for p in pages_b:
            fp    = doc_b[p]
            label = f"Page_{p+1}"
            if page_has_text(fp):
                grid = extract_text_page(str(pdf_path_b), p)
            else:
                img  = pdf_page_to_pil(fp, dpi=BATCH_DPI)
                img  = auto_rotate_image(img)
                grid = extract_table_grid_ocr(img)
            rows = len(grid)
            cols = max((len(r) for r in grid), default=0)
            print(f"  [{label}] {rows}r × {cols}c")
            grids_b.append((label, grid))

        doc_b.close()

        if BATCH_FORMAT == "xlsx":
            save_excel(grids_b, str(out_b))
        else:
            save_csv(grids_b, str(out_b))

    except Exception as exc:
        print(f"  ❌ ERROR: {exc}")
        continue

print("\n🎉 Batch complete!")
